In [ ]:
from pathlib import Path
import skimage 
import numpy as np 
import matplotlib as mpl
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
import torch
!pip install pytorch-msssim
from pytorch_msssim import ms_ssim

In [ ]:
!pip install lpips
import lpips

In [ ]:
PAS_paths =  ['Y:/coskun-lab/Ivan/Aditi Pix2pix/Notebooks/pytorch-CycleGAN-and-pix2pix/results/results_v2/sample_2_PAS_processed/test_latest/images', 
              'Y:/coskun-lab/Ivan/Aditi Pix2pix/Notebooks/pytorch-CycleGAN-and-pix2pix/results/results_v2/sample_3_PAS_processed/test_latest/images',
              'Y:/coskun-lab/Ivan/Aditi Pix2pix/Notebooks/pytorch-CycleGAN-and-pix2pix/results/results_v2/sample_4_PAS_processed/test_latest/images']
              
Trichrome_paths = ['Y:/coskun-lab/Ivan/Aditi Pix2pix/Notebooks/pytorch-CycleGAN-and-pix2pix/results/results_v2/sample_2_TRI_processed/test_latest/images', 
              'Y:/coskun-lab/Ivan/Aditi Pix2pix/Notebooks/pytorch-CycleGAN-and-pix2pix/results/results_v2/sample_3_TRI_processed/test_latest/images',
              'Y:/coskun-lab/Ivan/Aditi Pix2pix/Notebooks/pytorch-CycleGAN-and-pix2pix/results/results_v2/sample_4_TRI_processed/test_latest/images']

In [ ]:
PAS_image_paths = []
for PAS_path in PAS_paths: 
    PAS_image_paths.extend([f for f in Path(PAS_path).rglob('*.png')])

In [ ]:
Trichrome_image_paths = []
for Trichrome_path in Trichrome_paths: 
    Trichrome_image_paths.extend([f for f in Path(Trichrome_path).rglob('*.png')])

In [ ]:
real_PAS_images = []
virtual_PAS_images = []
for path in PAS_image_paths: 
    if('fake_B' in str(path)):
        virtual_PAS_images.append(skimage.io.imread(path))
    if('real_B' in str(path)):
        real_PAS_images.append(skimage.io.imread(path))

In [ ]:
real_Trichrome_images = []
virtual_Trichrome_images = []
for path in Trichrome_image_paths: 
    if('fake_B' in str(path)):
        virtual_Trichrome_images.append(skimage.io.imread(path))
    if('real_B' in str(path)):
        real_Trichrome_images.append(skimage.io.imread(path))

In [ ]:
def compute_ms_ssim(real_img, virtual_img):
    real = torch.tensor(real_img).permute(2,0,1).unsqueeze(0).float() / 255.
    virtual = torch.tensor(virtual_img).permute(2,0,1).unsqueeze(0).float() / 255.

    return ms_ssim(real, virtual, data_range=1.0).item()

In [ ]:
_lpips_model = lpips.LPIPS(net='alex')

def compute_lpips(real_img, virtual_img):
    real = torch.tensor(real_img).permute(2,0,1).unsqueeze(0).float() / 127.5 - 1
    virtual = torch.tensor(virtual_img).permute(2,0,1).unsqueeze(0).float() / 127.5 - 1

    with torch.no_grad():
        distance = _lpips_model(real, virtual)

    return distance.item()

In [ ]:
ssim_scores_PAS = []
ssim_scores_Trichrome = [] 

for i in range(len(real_PAS_images)):
    ssim_scores_PAS.append(ssim(real_PAS_images[i], virtual_PAS_images[i], channel_axis=-1))
    ssim_scores_Trichrome.append(ssim(real_Trichrome_images[i], virtual_Trichrome_images[i], channel_axis=-1))


In [ ]:
msi_ssim_scores_PAS = []
msi_ssim_scores_Trichrome = [] 

for i in range(len(real_PAS_images)):
    msi_ssim_scores_PAS.append(compute_ms_ssim(real_PAS_images[i], virtual_PAS_images[i]))
    msi_ssim_scores_Trichrome.append(compute_ms_ssim(real_Trichrome_images[i], virtual_Trichrome_images[i]))

In [ ]:
lpips_scores_PAS = []
lpips_scores_Trichrome = []

for i in range(len(real_PAS_images)):
    lpips_scores_PAS.append(compute_lpips(real_PAS_images[i], virtual_PAS_images[i]))
    lpips_scores_Trichrome.append(compute_lpips(real_Trichrome_images[i], virtual_Trichrome_images[i]))

In [ ]:
PAS_scores = [
    ssim_scores_PAS,
    msi_ssim_scores_PAS,
    lpips_scores_PAS
]

Trichrome_scores = [
    ssim_scores_Trichrome,
    msi_ssim_scores_Trichrome,
    lpips_scores_Trichrome
]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.size'] = 20

metrics = ["SSIM", "MS-SSIM", "LPIPS"]

palette = ["mediumvioletred", "royalblue"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (metric, pas, tri) in enumerate(zip(metrics, PAS_scores, Trichrome_scores)):
    
    sns.boxplot(
        data=[pas, tri],
        ax=axes[i],
        palette=palette,
        width=0.5
    )
    
    axes[i].set_xticklabels(["PAS", "Trichrome"])
    axes[i].set_title(metric)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Score")
    axes[i].set_ylim([0, 1])


plt.tight_layout()
plt.show()

In [ ]:
print(np.median(PAS_scores[0]))
print(np.median(PAS_scores[1]))
print(np.median(PAS_scores[2]))

print(np.median(Trichrome_scores[0]))
print(np.median(Trichrome_scores[1]))
print(np.median(Trichrome_scores[2]))

In [ ]:
print(np.percentile(PAS_scores[0], 75)-np.percentile(PAS_scores[0], 25))
print(np.percentile(PAS_scores[1], 75)-np.percentile(PAS_scores[1], 25))
print(np.percentile(PAS_scores[2], 75)-np.percentile(PAS_scores[2], 25))

print(np.percentile(Trichrome_scores[0], 75)-np.percentile(Trichrome_scores[0], 25))
print(np.percentile(Trichrome_scores[1], 75)-np.percentile(Trichrome_scores[1], 25))
print(np.percentile(Trichrome_scores[2], 75)-np.percentile(Trichrome_scores[2], 25))
